# Stage 1c — Checking the meanings we only guessed

**Purpose.** In stage 1b some column meanings were only **inferred [I]** from names and examples. A guess can be wrong, so here we test each one with code. The idea is simple: if a column means what we think, it must behave in a way we can predict, and we check that it does.

| Column | What we think it is | How we test it | What we expect if we are right |
|---|---|---|---|
| `related` | Links from an anime to other titles (sequel, prequel, adaptation...) | Parse the text into a table; check that a "Sequel" link is mirrored by a "Prequel" link | Most sequel links have a matching prequel link, and the linked ids exist in the anime table |
| `popularity` | A **rank** by number of members (1 = most popular) | Rank correlation with `members`, and with `score` as a control | Strongly **negative** with `members` (more members, smaller number), and clearly **weaker** with `score`, because being popular is not the same as being well rated |
| `rank` | A rank by `score` (1 = best score) | Rank correlation with `score` | Strongly **negative** |
| `members`, `scored_by`, `favorites` | Counts of users | Logical limits, and comparison with our own counts from the list table | `scored_by` and `favorites` never exceed `members`; our counts are proportional to the published ones |
| `score` | Average of the users' scores | Compare with the mean of the scores in our list table | Strong positive correlation |
| `episodes` | Number of episodes | Look at it by `type` | Movies have 1 episode; 0 may mean unknown |
| `stats_rewatched` | Something about rewatching, exact count unknown: anime rewatched, times rewatched, or episodes rewatched | Compare with the size of the user's list and with `stats_episodes`; compare with our own counts | Each reading makes a different prediction (see part 4) |
| `stats_episodes`, `stats_mean_score` | Total episodes watched, and the user's mean score | Recompute both from the user's rows in the list table | Close to the recomputed values |

**Important limit.** The list table covers only the users the author crawled (about 283,000), not everybody on MyAnimeList. So our own counts will be **smaller** than the published ones. We therefore look for strong agreement in the *order* of the anime (rank correlation) and a *stable ratio*, not for equal numbers.

**How to run on Kaggle.** Parts 1 and 2 take seconds. Part 3 reads two columns of the 80M-row list table (a minute or two); set `RUN_LISTS = False` in the setup cell to skip it.

In [ ]:
from pathlib import Path
import ast
import sys

import pandas as pd

# Works both on Kaggle and inside the repo (with the data placed in data/raw/)
KAGGLE_INPUT = Path("/kaggle/input")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "environment.yml").exists()), None)
if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
elif ROOT is not None:
    DATA_DIR = ROOT / "data" / "raw"
else:
    raise RuntimeError("Data not found: run on Kaggle or inside the repository.")

paths = {p.name: p for p in DATA_DIR.rglob("*.csv")}
RUN_LISTS = True  # False skips part 3, which reads 80M rows
na_kwargs = dict(keep_default_na=False, na_values=[""])  # only an empty cell is missing

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

anime = pd.read_csv(paths["AnimeList.csv"], **na_kwargs)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| anime:", anime.shape)

## 1. What is inside `related`?

Earlier the profile table cut every example at 45 characters, so only the first words were visible. Here we print three complete cells.

In [ ]:
for anime_id in anime.anime_id.head(3):
    print(anime_id, anime.loc[anime.anime_id == anime_id, "title"].iloc[0])
    print(anime.loc[anime.anime_id == anime_id, "related"].iloc[0], "\n")

The cell is the **text of a Python dictionary**. The keys are kinds of relation (for example `Adaptation`, `Sequel`), and each key holds a list of related titles. One cell can therefore describe several links, which is the "many values in one cell" problem from the tidy-data lesson. The code below turns it into a table with **one row per link**: `anime_id`, `relation`, `related_id`, `related_type`.

In [ ]:
def parse_related(text):
    """Turn the text of one `related` cell into a list of (relation, related_id, related_type)."""
    try:
        value = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return None  # the text cannot be parsed
    if value == []:
        return []  # an empty list means "no relation"
    if not isinstance(value, dict):
        return None
    rows = []
    for relation, items in value.items():
        for item in items:
            if isinstance(item, dict):
                rows.append((relation, item.get("mal_id"), item.get("type")))
    return rows


parsed = anime["related"].map(parse_related)
print("cells that could not be parsed:", int(parsed.isna().sum()))
print("anime whose cell holds no link:", int(parsed.map(lambda r: r == []).sum()))
print("what the unparsable cells look like:")
print(anime.loc[parsed.isna(), "related"].str.slice(0, 60).value_counts().head(5).to_string())

links = pd.DataFrame({"anime_id": anime.anime_id, "link": parsed}).dropna(subset=["link"]).explode("link")
links = links.dropna(subset=["link"])
link_cols = pd.DataFrame(links["link"].tolist(), index=links.index,
                         columns=["relation", "related_id", "related_type"])
relations = pd.concat([links[["anime_id"]], link_cols], axis=1).reset_index(drop=True)
relations["related_id"] = pd.to_numeric(relations["related_id"], errors="coerce")

print(f"\nlink rows: {len(relations):,} | anime with at least one link: {relations.anime_id.nunique():,}")
print("\nrelation types:")
print(relations["relation"].value_counts().to_string())
print("\nrelated_type (what the linked title is):")
print(relations["related_type"].value_counts(dropna=False).to_string())

In [ ]:
# Do the linked ids exist in the anime table? (only links to anime, not to manga)
anime_links = relations[relations["related_type"] == "anime"]
found = anime_links["related_id"].isin(anime["anime_id"])
print(f"links to anime: {len(anime_links):,} | target id found in the anime table: {found.mean():.1%}")

# Is a Sequel link mirrored by a Prequel link in the other direction?
sequel = relations.loc[relations["relation"] == "Sequel", ["anime_id", "related_id"]]
prequel = (relations.loc[relations["relation"] == "Prequel", ["anime_id", "related_id"]]
           .rename(columns={"anime_id": "related_id", "related_id": "anime_id"}))
check = sequel.merge(prequel, on=["anime_id", "related_id"], how="left", indicator=True)
print(f"Sequel links: {len(check):,} | mirrored by a Prequel link: {(check['_merge'] == 'both').mean():.1%}")

### Observations for part 1

**TODO (fill in after the run):** Were all cells parsable? Which relation types exist and how many links each? Which share of the linked ids is in the anime table, and which share of Sequel links is mirrored by a Prequel link? If the shares are high, the meaning of `related` is confirmed and we can use it to find sequels.

## 2. Do the audience columns behave like ranks and counts?

`popularity` and `rank` are expected to be ranks: **1 is the best**. A rank goes *down* when the thing it measures goes *up*, so the rank correlation with `members` (for `popularity`) and `score` (for `rank`) should be close to **−1**. Rank correlation (Spearman) compares the order of the values and ignores their size, which is what we want for a rank.

**A control test.** A correlation of −0.99 with `members` shows that `popularity` follows `members`, but it does not show that `popularity` is *only* about `members`. So we also correlate each rank with the *other* measure (`popularity` with `score`, and `rank` with `members`). If our reading is right, these cross correlations should be clearly weaker than the matching ones: an anime can have many members and a mediocre score, or few members and a high score. If they were just as strong, the two ranks would carry almost the same information and the matching correlation would prove less than it seems.

In [ ]:
def spearman(a, b):
    """Rank correlation: correlate the positions of the values, not the values themselves."""
    return a.rank().corr(b.rank())


print("popularity vs members:", round(spearman(anime["popularity"], anime["members"]), 3),
      "(close to -1 means popularity is a rank by members)")
ranked = anime.dropna(subset=["rank"])
print("rank vs score        :", round(spearman(ranked["rank"], ranked["score"]), 3),
      "(close to -1 means rank is a rank by score)")

# Control: each rank against the OTHER measure. Only anime with a real value in both columns.
cross = anime[(anime["popularity"] > 0) & (anime["score"] > 0)]
print("\ncontrol (should be clearly weaker than the matching correlations above):")
print("  popularity vs score  :", round(spearman(cross["popularity"], cross["score"]), 3),
      f"({len(cross):,} anime with popularity > 0 and score > 0)")
cross_rank = ranked[ranked["members"] > 0]
print("  rank vs members      :", round(spearman(cross_rank["rank"], cross_rank["members"]), 3),
      f"({len(cross_rank):,} anime with a rank and members > 0)")

print("\nlogical limits that must hold if the meanings are right:")
print("  scored_by > members :", int((anime["scored_by"] > anime["members"]).sum()), "anime")
print("  favorites > members :", int((anime["favorites"] > anime["members"]).sum()), "anime")
print("  score outside 0-10  :", int((~anime["score"].between(0, 10)).sum()), "anime")
print("  popularity is 0 or negative:", int((anime["popularity"] <= 0).sum()), "anime")
print("  popularity values unique:", anime["popularity"].is_unique,
      "| distinct:", anime["popularity"].nunique(), "of", len(anime))

In [ ]:
# episodes by type: do the numbers look like episode counts?
print(anime.groupby("type")["episodes"].agg(
    anime="size", zero=lambda s: int((s == 0).sum()), median="median", max="max").to_string())

### Observations for part 2

**TODO (fill in after the run):** Are both rank correlations close to −1? Are the two control correlations (`popularity` vs `score`, `rank` vs `members`) clearly weaker, and by how much? Does any logical limit fail, and for how many anime? Which types have `episodes == 0`, and does that look like "unknown"?

## 3. Compare the published numbers with our own counts

From the list table we can recompute, for every anime, how many of *our* users have it on their list (compare with `members`), how many of them scored it (compare with `scored_by`), and the mean of their scores (compare with `score`). The numbers will be smaller than the published ones because we only have a sample of the users, so we look at the **rank correlation** and at the **ratio** between our count and the published count.

In [ ]:
if RUN_LISTS:
    lists = pd.read_csv(paths["UserAnimeList.csv"], usecols=["anime_id", "my_score"],
                        dtype={"anime_id": "int32", "my_score": "int8"})
    rated = lists[lists["my_score"] > 0]
    ours = pd.DataFrame({
        "on_list_sample": lists.groupby("anime_id").size(),
        "rated_sample": rated.groupby("anime_id").size(),
        "mean_score_sample": rated.groupby("anime_id")["my_score"].mean(),
    })
    both = anime.set_index("anime_id")[["members", "scored_by", "score"]].join(ours, how="left")
    both[["on_list_sample", "rated_sample"]] = both[["on_list_sample", "rated_sample"]].fillna(0)
    print(f"list rows: {len(lists):,} | anime: {len(both):,} | with at least one list row:",
          int((both['on_list_sample'] > 0).sum()))

    def compare(published, sample, label):
        ok = both[(both[published] > 0) & (both[sample] > 0)]
        ratio = ok[sample] / ok[published]
        print(f"{label:34s} anime compared: {len(ok):>6,} | Spearman: {spearman(ok[published], ok[sample]):.3f}"
              f" | median ratio: {ratio.median():.3f} | middle 90% of ratios: {ratio.quantile(.05):.3f} to {ratio.quantile(.95):.3f}")

    compare("members", "on_list_sample", "members vs users with it on list")
    compare("scored_by", "rated_sample", "scored_by vs users who scored it")

    enough = both[(both["rated_sample"] >= 30) & (both["score"] > 0)]
    diff = enough["mean_score_sample"] - enough["score"]
    print(f"{'score vs mean score in sample':34s} anime compared: {len(enough):>6,} | Pearson: "
          f"{enough['score'].corr(enough['mean_score_sample']):.3f} | mean difference: {diff.mean():+.3f}"
          f" | middle 90%: {diff.quantile(.05):+.2f} to {diff.quantile(.95):+.2f}")
else:
    print("Skipped (RUN_LISTS = False)")

### Observations for part 3

**TODO (fill in after the run):** Is the rank correlation high for `members` and for `scored_by`? Is the ratio stable (a narrow middle range) or all over the place? Does the mean score in our sample follow `score`, and by how much does it differ on average? Stable ratios and high correlations confirm the meanings. Wild ratios mean a meaning is wrong or our sample is biased, and we must say which.

## 4. What do the per-user statistics count?

`stats_rewatched` can be read in three ways, and each reading predicts something different:

| Reading | Prediction we can test |
|---|---|
| (a) Number of **anime** the user has rewatched | Never larger than the number of anime on the user's list |
| (b) Total number of **times** the user rewatched anime | Can be larger than the list size, because one anime can be rewatched many times |
| (c) Number of rewatched **episodes** | Can be much larger than the list size, and cannot exceed `stats_episodes` |

The checks below only use the user table. Then the list table is used to recompute `stats_episodes` and `stats_mean_score` for every user. Even if the numbers fit one reading, the exact definition is MyAnimeList's, so what we can claim is "behaves like ...", not "is ...".

In [ ]:
users = pd.read_csv(paths["UserList.csv"], **na_kwargs)
count_cols = ["user_watching", "user_completed", "user_onhold", "user_dropped", "user_plantowatch"]
users["list_size"] = users[count_cols].sum(axis=1)  # anime on the list, from the user table itself
known = users.dropna(subset=["stats_rewatched"])

print("users:", len(users), "| with stats_rewatched:", len(known))
print("whole numbers only:", bool((known["stats_rewatched"] % 1 == 0).all()),
      "| negative values:", int((known["stats_rewatched"] < 0).sum()))
print(known["stats_rewatched"].describe(percentiles=[.5, .9, .99, .999]).round(1).to_string())
print("share of users with stats_rewatched = 0:", f"{(known['stats_rewatched'] == 0).mean():.1%}")

print("\nstats_rewatched > list size   :", int((known["stats_rewatched"] > known["list_size"]).sum()),
      "users   (0 would fit reading (a))")
print("stats_rewatched > stats_episodes:", int((known["stats_rewatched"] > known["stats_episodes"]).sum()),
      "users   (0 would fit reading (c))")

print("\nthe 10 largest values, next to the size of the user's list and their stats_episodes:")
print(known.nlargest(10, "stats_rewatched")[
    ["username", "stats_rewatched", "list_size", "user_completed", "stats_episodes"]].to_string(index=False))

In [ ]:
if RUN_LISTS:
    cols = {"username": "category", "my_score": "int8", "my_rewatching": "float32", "my_watched_episodes": "int32"}
    user_lists = pd.read_csv(paths["UserAnimeList.csv"], usecols=list(cols), dtype=cols, **na_kwargs)
    user_lists["rewatching_now"] = user_lists["my_rewatching"] == 1
    user_lists["score_if_rated"] = user_lists["my_score"].where(user_lists["my_score"] > 0)
    by_user = user_lists.groupby("username", observed=True)
    ours = pd.DataFrame({
        "rows": by_user.size(),
        "rewatching_rows": by_user["rewatching_now"].sum(),
        "episodes_sum": by_user["my_watched_episodes"].sum(),
        "mean_score_ours": by_user["score_if_rated"].mean(),
    })
    ours.index = ours.index.astype(object)
    joined = (users.set_index("username")[["stats_rewatched", "stats_episodes", "stats_mean_score", "list_size"]]
              .join(ours, how="inner").dropna(subset=["stats_rewatched"]))
    print(f"users with list rows and statistics: {len(joined):,}")

    def agreement(published, recomputed, label):
        ok = joined[(joined[published] > 0) & (joined[recomputed] > 0)]
        ratio = ok[recomputed] / ok[published]
        exact = (joined[published] == joined[recomputed]).mean()
        print(f"{label:46s} users compared: {len(ok):>7,} | Spearman: {spearman(ok[published], ok[recomputed]):.3f}"
              f" | exact equal (all users): {exact:.1%} | median ratio ours/published: {ratio.median():.3f}")

    agreement("stats_episodes", "episodes_sum", "stats_episodes vs sum of my_watched_episodes")
    agreement("stats_rewatched", "rewatching_rows", "stats_rewatched vs rows marked rewatching")
    agreement("stats_mean_score", "mean_score_ours", "stats_mean_score vs mean of our scores")
    diff = (joined["mean_score_ours"] - joined["stats_mean_score"]).dropna()
    print(f"mean score difference (ours - published): mean {diff.mean():+.3f}, "
          f"middle 90% from {diff.quantile(.05):+.2f} to {diff.quantile(.95):+.2f}, "
          f"within 0.05: {(diff.abs() <= 0.05).mean():.1%}")
else:
    print("Skipped (RUN_LISTS = False)")

### Observations for part 4

**TODO (fill in after the run):** Is `stats_rewatched` ever larger than the list size, and for how many users? Larger than `stats_episodes`? What do the ten largest values look like next to those users' list sizes? Which reading fits best, (a), (b) or (c), or none? Does the recomputed episode count and mean score agree with the published ones? Write only what the numbers show.

## After running

For every column whose behaviour matches the expectation in the table at the top, change its tag from **[I]** to **[V]** in the stage 1b dictionary and write the evidence (the correlation or share) next to it. For any column that does not behave as expected, correct the meaning and say what the output showed.